# R16c step 1 · R16's ceiling, and a counting check on the validation images (CPU, no GPU)

Task **R16c**, step 1, from `Essential_task_delegation/Progress_Tracker.md` (go given on 19 Oct 2026: test R16 more rigorously). It only reads saved outputs; R16, R16b and E3 stay unchanged.

**Why**
- R16 finds buildings with SAM 3 on the before-image, then grades each one. With the grader corrected (R16b), it still trails R11.
- Is the problem the outlines or the grader? Giving each found building its main true grade (a perfect grader) gives the ceiling of these outlines.
- Counting matters as much as pixels: one grade per building should make counting easier (O2). This checks that before any GPU goes to the benchmark (H1 V3).

**What it does**
1. **The ceiling (pixel scores, E-protocol):** R16's outlines graded four ways: R16's grader, R16b's corrected one, a perfect grader, and a perfect grader that may also reject non-buildings. Exact, from `instances_val.csv` and E1's validation masks.
2. **The counting check:**
   - The 1,055 training-split counting questions on our 645 validation images, the ones H1 V1 tuned on. DisasterM3's answer is the number of 8-connected blobs of the class, and the four wrong options sit ±20% and ±40% from it.
   - Each method counts blobs in its own map and picks the nearest option. The true masks must score 100%.
   - R16's maps are rebuilt image by image from E3's SAM 3 maps and R11's masks, and checked against `instances_val.csv`.
   - Nothing is tuned on these questions: R11 is counted with every blob, and once with R16's fixed 20 px minimum.

**Read with care:** none of these models trained on the 645 validation images, but validation numbers run higher than the benchmark's. R9's untuned counter scored about 33% on these questions, but its tuned counter scored 26.5% on the benchmark's optcal questions (H1 V1). Our 7B can't be scored here at all, because it trained on these images.

**Inputs on this PC** (all already here): `train_release.json`, the DisasterM3 masks in `dataset/` (for the split), `R16-results/R16/` (`instances_val.csv`, `R16_summary.csv`, `config.json`, `R11_tta/`), `R16b-results/config.json`, `E3-results/E3/masks/pre_grid2/` and `E1-results/E1/gt/`.

**Outputs in `Champion_Task/R16c-results/step1/`:** `R16c_ceiling.csv`, `R16c_counting.csv`, `R16c_counting_per_question.csv`, `config.json`, `README.md`. About 10 minutes.


In [ ]:
# ── Cell 1: Configuration and inputs (CPU only, no installs needed) ─────────
import json
import re
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image                # PIL and SciPy, as in D5b and R16b: this PC has no OpenCV
from scipy import ndimage
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# This PC: the repo root is the folder that holds train_release.json
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "train_release.json").exists())
CT = REPO / "Essential_task_delegation" / "Champion_Task"
DATA_ROOT = REPO / "dataset" / "DisasterM3 Dataset" / "DisasterM3_Instruct"
MANIFEST_PATH = REPO / "train_release.json"
R16_DIR = CT / "R16-results" / "R16"
SAM3_DIR = CT / "E3-results" / "E3" / "masks" / "pre_grid2"   # R16's finding: E3's SAM 3 instance maps (16-bit)
R11_DIR = R16_DIR / "R11_tta"                                  # R11's TTA masks (0-3), saved by R16
GT_DIR = CT / "E1-results" / "E1" / "gt"                       # the 645 validation masks (0-3), saved by E1
OUT_DIR = CT / "R16c-results" / "step1"

GRADES = ["Intact", "Damaged", "Destroyed"]
CLASS_NAMES = ["Background", *GRADES]
SOURCES = ("all", "xBD", "other")
R16_CONFIG = json.loads((R16_DIR / "config.json").read_text(encoding="utf-8"))
MIN_AREA = R16_CONFIG["min_area"]   # 20 px: R16's smallest building piece, also a fixed counting rule for R11
REAL_SHARE = np.array(json.loads((CT / "R16b-results" / "config.json").read_text(encoding="utf-8"))["real_share"])
CLASS_OF = {"Intact building counting.": 1, "Damaged building counting.": 2,
            "Totally-destroyed building counting.": 3}            # H1 V1's map of the counting questions
REFERENCES = {"our 7B, greedy, benchmark optical questions (H1 V2b)": 32.01,
              "base Qwen2.5-VL-7B, benchmark optical questions (paper)": 34.2,
              "R9's tuned counter, benchmark optical questions (H1 V1)": 26.54,
              "R9's untuned counter, these validation questions (H1 V1)": 33.3}

OUT_DIR.mkdir(parents=True, exist_ok=True)
t_start = time.time()
print(f"✓ R16c step 1 on this PC: {REPO}")
print(f"  SAM 3 maps: {len(list(SAM3_DIR.glob('*.png')))}, R11 masks: {len(list(R11_DIR.glob('*.png')))}, "
      f"validation masks: {len(list(GT_DIR.glob('*.png')))} (645 each expected)")
print(f"  R16b's real shares: {dict(zip(GRADES, REAL_SHARE.round(4)))}; outputs: {OUT_DIR}")

✓ R16c step 1 on this PC: D:\CSE499AB_project
  SAM 3 maps: 645, R11 masks: 645, validation masks: 645 (645 each expected)
  R16b's real shares: {'Intact': np.float64(0.7368), 'Damaged': np.float64(0.1836), 'Destroyed': np.float64(0.0796)}; outputs: D:\CSE499AB_project\Essential_task_delegation\Champion_Task\R16c-results\step1


In [2]:
# ── Cell 2: R16's validation split, and the counting questions on its 645 images ──
with open(MANIFEST_PATH, encoding="utf-8") as f:
    raw = json.load(f)

def resolve_mask(rel_path):
    """The mask candidates of the D4 notebooks' resolve_path (this PC holds the masks, not the images)."""
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    for c in (DATA_ROOT / rel_path, DATA_ROOT / "masks" / rel_path, DATA_ROOT / "masks" / "masks" / Path(rel_path).name):
        if c.exists():
            return c
    return None

def name_of(path):
    return Path(str(path).replace("\\", "/")).name

# R16's pairs, in manifest order (as in R16b), then its split
by_image = defaultdict(dict)
for e in raw:
    if (e.get("task") == "Referring Expression Segmentation" and e.get("cls_description") == "Building Damage Assessment"
            and e.get("image_type") == "Optical"):
        rel = e.get("ground_truth", "")
        path = resolve_mask(rel)
        if path:
            by_image[e.get("post_image_path", "")][Path(rel.replace("\\", "/")).parent.name] = path
names = list(by_image)
assert len(names) == R16_CONFIG["train_images"] + R16_CONFIG["val_images"], f"{len(names):,} images; R16 had 6,443"
_, val_names = train_test_split(names, test_size=0.1, random_state=42)
val_files = sorted(name_of(n) for n in val_names)
assert set(val_files) == {p.name for p in GT_DIR.glob("*.png")}, "E1's masks aren't R16's 645 validation images"

# The training-split counting questions on these images (H1 V1's tuning set)
val_set = set(val_files)
questions = [e for e in raw if e.get("task") == "Building Damage Counting" and name_of(e["post_image_path"]) in val_set]
del raw
assert len(questions) == 1055, f"{len(questions):,} counting questions on the validation images; H1 V1 found 1,055"

XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")   # R16's rule: xBD tiles, everything else is "other"

def source_of(name):
    return "xBD" if XBD_NAME.match(name) else "other"

by_class = pd.Series([e["cls_description"] for e in questions]).value_counts().to_dict()
print(f"✓ {len(val_files)} validation images; {len(questions):,} counting questions on them: {by_class}")

✓ 645 validation images; 1,055 counting questions on them: {'Intact building counting.': 555, 'Damaged building counting.': 284, 'Totally-destroyed building counting.': 216}


## 1. The ceiling: R16's outlines with a perfect grader
Each found building is painted with one class. The confusion matrix is every building's true pixel counts added to the column of its class, plus the remaining true pixels in the background column (exact, as R16b showed). The maps:
- **SAM 3 only** or **SAM 3 + R11's pieces** (the buildings SAM 3 missed), each with four sets of grades:
  - R16's grader;
  - R16b's corrected grader;
  - a **perfect grader**: each building's main true grade;
  - a **perfect grader + rejects**: it may also call a building background when most of its pixels are.
- **The vote** with R11 (R16's third map), with R16's and R16b's grades.

In [ ]:
# ── Cell 3: The ceiling, from instances_val.csv and the validation masks ─────
totals = {s: np.zeros(4, np.int64) for s in SOURCES}      # true pixels per class
for name in tqdm(val_files, desc="Validation masks"):
    counts = np.bincount(np.array(Image.open(GT_DIR / name).convert("L")).ravel(), minlength=4)
    assert len(counts) == 4, f"{name} has values above 3"
    totals["all"] += counts
    totals[source_of(name)] += counts

inst = pd.read_csv(R16_DIR / "instances_val.csv")
GT_COLS = [f"gt_{c.lower()}_px" for c in CLASS_NAMES]
R11_COLS = [f"r11_{c.lower()}_px" for c in CLASS_NAMES]
P = inst[[f"p_{g.lower()}" for g in GRADES]].to_numpy(np.float64)
Pb = P * REAL_SHARE                 # R16b: each grade times its real share (the training share was a third)
Pb /= Pb.sum(1, keepdims=True)
gt_px = inst[GT_COLS].to_numpy(np.int64)
r11_bld = inst[R11_COLS[1:]].to_numpy(np.float64)
r11_sum = r11_bld.sum(1, keepdims=True)
is_sam = (inst["stage1"] == "sam3").to_numpy()

def vote(probs):
    """R16's third map: the average of the grader and R11's pixel shares inside the building."""
    shares = np.where(r11_sum > 0, r11_bld / np.maximum(r11_sum, 1), probs)
    return (0.5 * (probs + shares)).argmax(1) + 1

GRADE_SETS = {"R16": P.argmax(1) + 1, "R16b": Pb.argmax(1) + 1,
              "perfect": gt_px[:, 1:].argmax(1) + 1,   # the main true grade (intact if it has none)
              "perfect + rejects": gt_px.argmax(1)}    # may also say background
PAINT = {}                                             # map name -> class (0-3) of each found building
for g, cls in GRADE_SETS.items():
    PAINT[f"SAM 3 only, {g} grades"] = np.where(is_sam, cls, 0)
    PAINT[f"SAM 3 + R11 pieces, {g} grades"] = cls
PAINT["vote with R11, R16 grades"] = vote(P)
PAINT["vote with R11, R16b grades"] = vote(Pb)

def confusion(rows_gt_px, building_class, total):
    """4×4 pixel counts of one map (rows truth, columns prediction)."""
    cm = np.zeros((4, 4), np.int64)
    for k in range(4):
        cm[:, k] += rows_gt_px[building_class == k].sum(0)
    cm[:, 0] += total - rows_gt_px.sum(0)      # outside every found building: background
    return cm

def scores(cm):
    cm = cm.astype(np.float64)
    tp = np.diag(cm)
    iou = tp / (cm.sum(0) + cm.sum(1) - tp + 1e-8)
    building_tp = cm[1:, 1:].sum()
    building_iou = building_tp / (building_tp + cm[0, 1:].sum() + cm[1:, 0].sum() + 1e-8)
    return {"miou": round(float(iou.mean()), 4),
            **{f"{c.lower()}_iou": round(float(iou[i]), 4) for i, c in enumerate(CLASS_NAMES)},
            "building_iou": round(float(building_iou), 4)}

r16_summary = pd.read_csv(R16_DIR / "R16_summary.csv")
ceiling = []
for source in SOURCES:
    sel = np.ones(len(inst), bool) if source == "all" else (inst["source"] == source).to_numpy()
    ref = r16_summary[(r16_summary["variant"] == "R11_tta") & (r16_summary["source"] == source)].iloc[0]
    ceiling.append({"source": source, "map": "R11 with TTA (reference)",
                    **{k: ref[k] for k in ("miou", "background_iou", "intact_iou", "damaged_iou", "destroyed_iou",
                                           "building_iou")}})
    for map_name, cls in PAINT.items():
        ceiling.append({"source": source, "map": map_name, **scores(confusion(gt_px[sel], cls[sel], totals[source]))})
ceiling = pd.DataFrame(ceiling)

# The rebuilt R16 and R16b rows must match their own results files
check = {("SAM 3 only, R16 grades", "R16_sam3"), ("SAM 3 + R11 pieces, R16 grades", "R16_sam3+r11"),
         ("vote with R11, R16 grades", "R16_vote")}
for map_name, variant in check:
    mine = ceiling[(ceiling["map"] == map_name) & (ceiling["source"] == "all")].iloc[0]
    theirs = r16_summary[(r16_summary["variant"] == variant) & (r16_summary["source"] == "all")].iloc[0]
    assert abs(mine["miou"] - theirs["miou"]) < 1e-4 and abs(mine["destroyed_iou"] - theirs["destroyed_iou"]) < 1e-4, map_name
show = ["map", "miou", "intact_iou", "damaged_iou", "destroyed_iou", "building_iou"]
print("✓ R16's rows rebuilt exactly. The ceiling on all 645 images:")
print(ceiling[ceiling["source"] == "all"][show].to_string(index=False))

Validation masks:   0%|          | 0/645 [00:00<?, ?it/s]

✓ R16's rows rebuilt exactly. The ceiling on all 645 images:
                                         map   miou  intact_iou  damaged_iou  destroyed_iou  building_iou
                    R11 with TTA (reference) 0.5875      0.6821       0.3956         0.2965        0.7420
                      SAM 3 only, R16 grades 0.4711      0.4713       0.2772         0.1672        0.6304
              SAM 3 + R11 pieces, R16 grades 0.4941      0.5690       0.2800         0.1522        0.7426
                     SAM 3 only, R16b grades 0.5150      0.5450       0.3186         0.2280        0.6304
             SAM 3 + R11 pieces, R16b grades 0.5407      0.6441       0.3356         0.2080        0.7426
                  SAM 3 only, perfect grades 0.7060      0.6243       0.6755         0.5555        0.6304
          SAM 3 + R11 pieces, perfect grades 0.7657      0.7382       0.7359         0.6136        0.7426
        SAM 3 only, perfect + rejects grades 0.7164      0.6414       0.6793         0.5742

## 2. The counting check
For each validation image, R16's finding map is rebuilt as R16's Cell 9 built it: SAM 3's instances, then R11's building pieces of at least 20 px that SAM 3 missed. Every rebuilt building must match a row of `instances_val.csv` exactly (its true and R11 pixel counts per class), which also gives it its grader probabilities. Each map is painted, its blobs are counted per class, and the nearest option answers the question.

In [ ]:
# ── Cell 4: Rebuild each image's finding map, check it, and count blobs per method ──
EIGHT = np.ones((3, 3), bool)

def blobs(binary, min_area=0):
    """Number of 8-connected blobs (DisasterM3's counting rule); min_area > 0 drops smaller ones."""
    labels, n = ndimage.label(binary, structure=EIGHT)
    if min_area == 0 or n == 0:
        return n
    return int((np.bincount(labels.ravel())[1:] >= min_area).sum())

# Self-checks: a diagonal touch is one blob; the area rule drops small blobs
_m = np.zeros((6, 6), bool)
_m[1, 1] = _m[2, 2] = True
_m[4, 3:6] = True
assert blobs(_m) == 2 and blobs(_m, min_area=3) == 1

METHODS = ["truth", "R11, every blob", f"R11, blobs of at least {MIN_AREA} px", *PAINT]
SIG_COLS = GT_COLS + R11_COLS
groups = {name: rows for name, rows in inst.groupby("image")}
counts = {}                          # (image, method) -> [intact, damaged, destroyed]
unmatched, ambiguous = 0, 0
for name in tqdm(val_files, desc="Rebuild and count"):
    stem = Path(name).stem
    gt = np.array(Image.open(GT_DIR / name).convert("L"))
    r11 = np.array(Image.open(R11_DIR / f"{stem}.png").convert("L"))
    sam_path = SAM3_DIR / f"{stem}.png"
    sam = np.array(Image.open(sam_path)).astype(np.int64) if sam_path.exists() else np.zeros(gt.shape, np.int64)
    assert sam.shape == r11.shape == gt.shape, f"{name}: map sizes differ"

    # R16's finding: SAM 3's ids, then R11's pieces (8-connected, at least MIN_AREA) where SAM 3 found nothing
    n_sam = int(sam.max())
    extra, n_extra = ndimage.label((r11 > 0) & (sam == 0), structure=EIGHT)
    areas = np.bincount(extra.ravel(), minlength=n_extra + 1)
    keep = areas >= MIN_AREA
    keep[0] = False
    new_id = np.zeros(n_extra + 1, np.int64)
    new_id[keep] = n_sam + 1 + np.arange(int(keep.sum()))
    ids = np.where(extra > 0, new_id[extra], sam)

    # Match every rebuilt building to its row: SAM 3's by id, R11's pieces by their pixel counts
    # (R16 numbered R11's pieces with OpenCV, whose order may differ from SciPy's)
    size = int(ids.max()) + 1
    flat = ids.ravel() * 4
    sig = np.hstack([np.bincount(flat + gt.ravel(), minlength=size * 4).reshape(size, 4),
                     np.bincount(flat + r11.ravel(), minlength=size * 4).reshape(size, 4)])
    present = np.flatnonzero(sig[:, :4].sum(1))
    present = present[present > 0]
    row_of = np.full(size, -1, np.int64)
    rows = groups.get(name)
    if rows is not None:
        sam_rows = rows[rows["stage1"] == "sam3"]
        row_of[sam_rows["id"].to_numpy()] = sam_rows.index.to_numpy()
        unmatched += int((sig[sam_rows["id"].to_numpy()] != sam_rows[SIG_COLS].to_numpy()).any(1).sum())
        by_sig = defaultdict(list)
        for i, s in zip(rows.index[rows["stage1"] != "sam3"], rows.loc[rows["stage1"] != "sam3", SIG_COLS].to_numpy()):
            by_sig[tuple(s.tolist())].append(i)
        ambiguous += sum(len(v) - 1 for v in by_sig.values())
        for pid in present[present > n_sam]:
            hits = by_sig.get(tuple(sig[pid].tolist()))
            row_of[pid] = hits.pop() if hits else -1
    unmatched += int((row_of[present] < 0).sum())

    counts[(name, "truth")] = [blobs(gt == c) for c in (1, 2, 3)]
    counts[(name, "R11, every blob")] = [blobs(r11 == c) for c in (1, 2, 3)]
    counts[(name, f"R11, blobs of at least {MIN_AREA} px")] = [blobs(r11 == c, MIN_AREA) for c in (1, 2, 3)]
    for method, building_class in PAINT.items():
        lut = np.zeros(size, np.uint8)
        found = row_of >= 0
        lut[found] = building_class[row_of[found]]
        painted = lut[ids]
        counts[(name, method)] = [blobs(painted == c) for c in (1, 2, 3)]

assert unmatched == 0, f"{unmatched} rebuilt buildings don't match instances_val.csv: R16's finding isn't reproduced"
print(f"✓ Rebuilt {len(val_files)} finding maps; every building matches instances_val.csv "
      f"({ambiguous} pairs of R11 pieces with identical pixel counts); {(time.time() - t_start) / 60:.1f} min so far")

Rebuild and count:   0%|          | 0/645 [00:00<?, ?it/s]

✓ Rebuilt 645 stage-1 maps; every building matches instances_val.csv (526 pairs of R11 pieces with identical pixel counts); 2.6 min so far


In [ ]:
# ── Cell 5: Answer the questions, score each method, and write the outputs ──
def nearest_option(n, options):
    """H1 V1's rule: the option closest to the count (the first one on a tie)."""
    return "ABCDE"[min(range(len(options)), key=lambda i: abs(options[i] - n))]

assert nearest_option(52, [10, 40, 60, 80, 100]) == "C"
rows = []
for e in questions:
    name, cls = name_of(e["post_image_path"]), CLASS_OF[e["cls_description"]]
    row = {"image": name, "source": source_of(name), "class": GRADES[cls - 1].lower(),
           "truth": e["ground_truth"], "truth_option": e["ground_truth_option"]}
    for m in METHODS:
        n = counts[(name, m)][cls - 1]
        row[f"{m} | count"] = n
        row[f"{m} | correct"] = nearest_option(n, e["options_list"]) == e["ground_truth_option"]
    rows.append(row)
per_q = pd.DataFrame(rows)
per_q.to_csv(OUT_DIR / "R16c_counting_per_question.csv", index=False)

def accuracy(df, m):
    return round(100 * df[f"{m} | correct"].mean(), 2) if len(df) else float("nan")

table = []
for m in METHODS:
    err = ((per_q[f"{m} | count"] - per_q["truth"]).abs() / per_q["truth"].clip(lower=1))
    table.append({"method": m, "all_%": accuracy(per_q, m),
                  **{f"{c}_%": accuracy(per_q[per_q["class"] == c], m) for c in ("intact", "damaged", "destroyed")},
                  **{f"{s}_%": accuracy(per_q[per_q["source"] == s], m) for s in ("xBD", "other")},
                  "median_count_error_%": round(100 * float(err.median()), 1)})
counting = pd.DataFrame(table)
counting.to_csv(OUT_DIR / "R16c_counting.csv", index=False)
ceiling.to_csv(OUT_DIR / "R16c_ceiling.csv", index=False)
truth_acc = counting.loc[counting["method"] == "truth", "all_%"].item()
assert truth_acc == 100.0, f"the true masks answer {truth_acc}% of the questions, not 100%: the counting rule is off"
print(counting.to_string(index=False))

# What it says, stated against R11 on the same questions (the only like-for-like comparison here)
real = [m for m in PAINT if "perfect" not in m]
best = max(real, key=lambda m: counting.loc[counting["method"] == m, "all_%"].item())
best_acc = counting.loc[counting["method"] == best, "all_%"].item()
r11_best = max(counting.loc[counting["method"].str.startswith("R11"), "all_%"])
ceiling_acc = max(counting.loc[counting["method"].str.contains("perfect"), "all_%"])
finding = (f"Best R16 map for counting: '{best}', {best_acc:.1f}% of the 1,055 validation questions, against "
           f"{r11_best:.1f}% for R11's blobs ({best_acc - r11_best:+.1f} points) and {ceiling_acc:.1f}% with a perfect "
           f"grader on the same outlines.")
print(f"\n{finding}")
print("For scale (other images, so only a guide): " + "; ".join(f"{k} {v}%" for k, v in REFERENCES.items()))

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

config = {"task": "R16c step 1", "validation_images": len(val_files), "questions": len(per_q),
          "stage1_buildings": len(inst), "r11_pieces_with_identical_counts": ambiguous, "min_area": MIN_AREA,
          "real_share": REAL_SHARE.round(4).tolist(), "truth_accuracy_%": truth_acc, "finding": finding,
          "references_%": REFERENCES, "run_time_min": round((time.time() - t_start) / 60, 1)}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    "# R16c step 1: R16's ceiling and a counting check", "",
    "From saved outputs only (R16, R16b, E3, E1), on the 645 validation images. Nothing is tuned on them.", "",
    "## 1. The ceiling (pixel scores, all 645 images)", "",
    "A perfect grader gives each found building its main true grade; with rejects it may also call a building "
    "background. The gap between a real grader and the perfect one on the same outlines is what a better grader "
    "can win.", "",
    as_markdown(ceiling[ceiling["source"] == "all"][show]), "",
    "Per source (xBD and other) in `R16c_ceiling.csv`.", "",
    "## 2. Counting", "",
    f"The {len(per_q):,} training-split counting questions on the validation images. DisasterM3's answer is the number "
    "of 8-connected blobs of the class; each method counts blobs in its own map and picks the nearest option. "
    f"The true masks answer {truth_acc:.0f}%.", "",
    as_markdown(counting), "",
    f"**Finding:** {finding}", "",
    "**For scale** (other images, so only a guide; validation numbers run high: R9's untuned counter scored about "
    "33% here but its tuned counter 26.5% on the benchmark's optical questions):", "",
    *[f"- {k}: {v}%" for k, v in REFERENCES.items()], "",
    "Our 7B can't be scored here: it trained on these images.", "",
    "## Files", "",
    "- `R16c_ceiling.csv`: pixel scores of every map, overall and per source",
    "- `R16c_counting.csv` and `R16c_counting_per_question.csv`: the counting check",
    "- `config.json`",
]), encoding="utf-8")
print(f"\n✓ Wrote R16c_ceiling.csv, R16c_counting.csv, R16c_counting_per_question.csv, config.json and README.md "
      f"to {OUT_DIR} ({(time.time() - t_start) / 60:.1f} min)")

                                      method  all_%  intact_%  damaged_%  destroyed_%  xBD_%  other_%  median_count_error_%
                                       truth 100.00    100.00     100.00       100.00 100.00   100.00                   0.0
                             R11, every blob  37.16     36.76      37.32        37.96  39.50    21.99                  33.3
                R11, blobs of at least 20 px  39.24     38.02      43.66        36.57  42.67    17.02                  33.3
                      SAM 3 only, R16 grades  30.14     29.55      33.10        27.78  31.51    21.28                  50.0
              SAM 3 + R11 pieces, R16 grades  31.37     34.41      37.68        15.28  32.60    23.40                  40.0
                     SAM 3 only, R16b grades  36.40     36.76      34.86        37.50  38.18    24.82                  36.7
             SAM 3 + R11 pieces, R16b grades  34.41     38.56      34.51        23.61  37.31    15.60                  33.3
        